# Train 5 classifiers (Logistic, DecisionTree, KNN, GaussianNB, RandomForest)

This notebook trains the required pipelines, exports `model/*.pkl`, `metrics.json`, `schema.json`, and `../test_data.csv`. Follow the assignment `CLAUDE_1.md`.

In [1]:
# Imports and config
RANDOM_STATE = 42
import json
from pathlib import Path
import pandas as pd
import numpy as np
import sklearn
ROOT = Path('..').resolve()
print('scikit-learn', sklearn.__version__)
print('ROOT:', ROOT)

scikit-learn 1.9.0
ROOT: C:\Joseph_Vinod\dev\bits-ml-assignment-2


In [2]:
# Load and inspect
DATA_PATH = Path('..') / 'data' / 'original_data.csv'
df = pd.read_csv(DATA_PATH, sep=None, engine='python')
print('shape:', df.shape)
print('dtypes:')
print(df.dtypes.value_counts())
print('missing per column:')
print(df.isna().sum().sort_values(ascending=False).head(10))
if 'Target' in df.columns:
    print('Target distribution:')
    print(df['Target'].value_counts())
else:
    raise SystemExit('Target column not found; aborting')

shape: (4424, 37)
dtypes:
int64      29
float64     7
str         1
Name: count, dtype: int64
missing per column:
﻿Marital status                   0
Application mode                  0
Application order                 0
Course                            0
Daytime/evening attendance\t      0
Previous qualification            0
Previous qualification (grade)    0
Nacionality                       0
Mother's qualification            0
Father's qualification            0
dtype: int64
Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


In [3]:
# Column typing (detect nominal integer-coded columns)
FEATURE_COLUMNS = [c for c in df.columns if c != 'Target']
numeric_cols = [c for c in df.select_dtypes(include=['number']).columns if c != 'Target']
nominal_candidates = []
for c in numeric_cols:
    if pd.api.types.is_integer_dtype(df[c]) and df[c].nunique(dropna=True) <= 30:
        nominal_candidates.append(c)
NOMINAL_COLS = nominal_candidates
NUMERIC_COLS = [c for c in FEATURE_COLUMNS if c not in NOMINAL_COLS]
print('NUMERIC_COLS count:', len(NUMERIC_COLS))
print('NOMINAL_COLS count:', len(NOMINAL_COLS))
print('NOMINAL_EXAMPLE', NOMINAL_COLS[:20])

NUMERIC_COLS count: 12
NOMINAL_COLS count: 24
NOMINAL_EXAMPLE ['\ufeffMarital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance\t', 'Previous qualification', 'Nacionality', "Mother's qualification", 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)']


In [4]:
# Split (stratified) and export test set
from sklearn.model_selection import train_test_split
X = df[FEATURE_COLUMNS].copy()
y = df['Target'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)
output_test = Path('..') / 'test_data.csv'
test_df.to_csv(output_test, index=False)
print('Wrote test_data.csv with shape', test_df.shape)

Wrote test_data.csv with shape (1106, 37)


In [5]:
# Preprocessor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
num_pipeline = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())])
cat_pipeline = Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUMERIC_COLS),
    ('cat', cat_pipeline, NOMINAL_COLS),
])
print('Preprocessor created')

Preprocessor created


In [6]:
# Model definitions and training loop
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'decision_tree': DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=RANDOM_STATE),
    'knn': KNeighborsClassifier(n_neighbors=15),
    'naive_bayes': GaussianNB(var_smoothing=1e-2),
    'random_forest': RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1),
}
Path('..' + '/model').mkdir(parents=True, exist_ok=True)
metrics = {}
display_name_overrides = {'naive_bayes': 'Gaussian Naive Bayes'}
def compute_metrics(y_true, y_pred, y_proba, classes):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro', labels=classes)
    return {'accuracy': float(acc), 'precision': float(prec), 'recall': float(rec), 'f1': float(f1), 'mcc': float(mcc), 'auc': float(auc)}

for slug, clf in models.items():
    print('Training', slug)
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)
    m = compute_metrics(y_test, y_pred, y_proba, classes=pipe.classes_)
    metrics[slug] = {'display_name': display_name_overrides.get(slug, slug.replace('_', ' ').title()), **m}
    joblib.dump(pipe, Path('..') / 'model' / f'{slug}.pkl')
    print('Saved', slug, '-> model/' + slug + '.pkl')

# Write metrics.json and schema.json
with open(Path('..') / 'model' / 'metrics.json', 'w', encoding='utf8') as f:
    json.dump(metrics, f, indent=2)
schema = {
    'target_column': 'Target',
    'feature_columns': FEATURE_COLUMNS,
    'class_labels': sorted(df['Target'].unique().tolist()),
    'nominal_columns': NOMINAL_COLS,
    'sklearn_version': sklearn.__version__,
    'n_train': int(X_train.shape[0]),
    'n_test': int(X_test.shape[0]),
}
with open(Path('..') / 'model' / 'schema.json', 'w', encoding='utf8') as f:
    json.dump(schema, f, indent=2)
print('Wrote metrics.json and schema.json')

Training logistic_regression
Saved logistic_regression -> model/logistic_regression.pkl
Training decision_tree
Saved decision_tree -> model/decision_tree.pkl
Training knn
Saved knn -> model/knn.pkl
Training naive_bayes
Saved naive_bayes -> model/naive_bayes.pkl
Training random_forest
Saved random_forest -> model/random_forest.pkl
Wrote metrics.json and schema.json
